# ASC-Höhenmodell nach Web Mercator GeoTIFF konvertieren

Dieses Notebook liest ein ESRI-ASCII-Grid (`.asc`), schreibt es zunächst als GeoTIFF mit Quell-Koordinatensystem `EPSG:21781` und transformiert es anschliessend nach **Web Mercator** (`EPSG:3857`).

Du musst nur im ersten Code-Block die Pfade zu deiner Eingabe- und Ausgabedatei anpassen.

## 1. Bibliotheken installieren

Falls `rasterio` oder `numpy` noch nicht installiert sind, führe diese Zelle einmal aus.

In [ ]:
# Falls nötig aktivieren:
# %pip install rasterio numpy

## 2. Input- und Output-Dateien definieren

Passe hier die Pfade an. Es ist keine Eingabe im Terminal nötig.

In [1]:
from pathlib import Path

# Eingabe: deine ASC-Datei
input_asc = Path("DHM200.asc")

# Ausgabe: GeoTIFF in Web Mercator
output_tif = Path("dhm_webmercator.tif")

# Koordinatensysteme
src_crs = "EPSG:21781"   # CH1903 / LV03
dst_crs = "EPSG:3857"   # Web Mercator

print("Input:", input_asc)
print("Output:", output_tif)
print("Quelle:", src_crs)
print("Ziel:", dst_crs)

Input: DHM200.asc
Output: dhm_webmercator.tif
Quelle: EPSG:21781
Ziel: EPSG:3857


## 3. Funktionen definieren

Diese Funktionen lesen das ASC-Grid, schreiben ein temporäres GeoTIFF und reprojizieren es nach Web Mercator.

In [2]:
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling


def read_ascii_grid(path):
    """
    Liest ein ESRI ASCII Grid (.asc) robust ein.

    Manche ASC-Dateien speichern eine Rasterzeile nicht zwingend als eine Textzeile.
    Deshalb wird der Datenblock als fortlaufender Zahlenstrom gelesen und danach
    anhand von nrows und ncols neu geformt.
    """
    path = Path(path)
    header = {}

    with open(path, "r") as f:
        for _ in range(6):
            line = f.readline()
            parts = line.strip().split()

            if len(parts) < 2:
                raise ValueError(f"Ungültige Header-Zeile: {line!r}")

            key = parts[0].lower()
            value = float(parts[1])
            header[key] = value

        ncols = int(header["ncols"])
        nrows = int(header["nrows"])
        cellsize = header["cellsize"]
        nodata = header.get("nodata_value", -9999)

        values = np.fromfile(f, sep=" ", dtype=np.float32)

    expected_values = nrows * ncols

    if values.size != expected_values:
        raise ValueError(
            f"Anzahl Rasterwerte passt nicht zum Header.\n"
            f"Erwartet: {expected_values} Werte ({nrows} × {ncols})\n"
            f"Gefunden: {values.size} Werte\n"
            f"Prüfe, ob die Datei wirklich ein ESRI ASCII Grid ist "
            f"oder ob zusätzliche Zeilen/Kommentare im Datenblock enthalten sind."
        )

    data = values.reshape((nrows, ncols))

    if "xllcorner" in header:
        x_min = header["xllcorner"]
    elif "xllcenter" in header:
        x_min = header["xllcenter"] - cellsize / 2
    else:
        raise ValueError("Header enthält weder xllcorner noch xllcenter.")

    if "yllcorner" in header:
        y_min = header["yllcorner"]
    elif "yllcenter" in header:
        y_min = header["yllcenter"] - cellsize / 2
    else:
        raise ValueError("Header enthält weder yllcorner noch yllcenter.")

    y_max = y_min + nrows * cellsize
    transform = from_origin(x_min, y_max, cellsize, cellsize)

    return data, transform, nodata


def write_geotiff(path, data, transform, crs, nodata):
    """Schreibt ein einzelnes Band als GeoTIFF."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs=crs,
        transform=transform,
        nodata=nodata,
        compress="lzw",
        tiled=True,
        BIGTIFF="IF_SAFER",
    ) as dst:
        dst.write(data.astype(np.float32), 1)


def reproject_raster(input_tif, output_tif, dst_crs):
    """Reprojiziert ein GeoTIFF in ein Ziel-Koordinatensystem."""
    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(input_tif) as src:
        transform, width, height = calculate_default_transform(
            src.crs,
            dst_crs,
            src.width,
            src.height,
            *src.bounds,
        )

        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height,
            "compress": "lzw",
            "tiled": True,
            "BIGTIFF": "IF_SAFER",
        })

        with rasterio.open(output_tif, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                src_nodata=src.nodata,
                dst_nodata=src.nodata,
                resampling=Resampling.bilinear,
            )


def asc_to_webmercator_geotiff(input_asc, output_tif, src_crs="EPSG:21781", dst_crs="EPSG:3857"):
    """
    Kompletter Workflow:
    1. ASC lesen
    2. temporäres GeoTIFF schreiben
    3. nach Web Mercator reprojizieren

    Windows-Fix:
    Es wird bewusst TemporaryDirectory statt NamedTemporaryFile verwendet,
    weil Windows geöffnete temporäre Dateien für andere Prozesse sperrt.
    """
    input_asc = Path(input_asc)
    output_tif = Path(output_tif)

    if not input_asc.exists():
        raise FileNotFoundError(f"Input-Datei nicht gefunden: {input_asc}")

    data, transform, nodata = read_ascii_grid(input_asc)

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_tif = Path(tmp_dir) / "intermediate_epsg21781.tif"

        write_geotiff(
            path=tmp_tif,
            data=data,
            transform=transform,
            crs=src_crs,
            nodata=nodata,
        )

        reproject_raster(
            input_tif=tmp_tif,
            output_tif=output_tif,
            dst_crs=dst_crs,
        )

    return output_tif


## 4. Konvertierung starten

Führe diese Zelle aus, nachdem du oben `input_asc` und `output_tif` angepasst hast.

In [3]:
result = asc_to_webmercator_geotiff(
    input_asc=input_asc,
    output_tif=output_tif,
    src_crs=src_crs,
    dst_crs=dst_crs,
)

print(f"Fertig geschrieben: {result}")

Fertig geschrieben: dhm_webmercator.tif


## 5. Ergebnis prüfen

Diese Zelle zeigt grundlegende Informationen zum erzeugten GeoTIFF.

In [4]:
with rasterio.open(output_tif) as ds:
    print("Datei:", output_tif)
    print("CRS:", ds.crs)
    print("Breite x Höhe:", ds.width, "x", ds.height)
    print("Bounds:", ds.bounds)
    print("NoData:", ds.nodata)
    print("Transform:", ds.transform)

Datei: dhm_webmercator.tif
CRS: EPSG:3857
Breite x Höhe: 1963 x 1232
Bounds: BoundingBox(left=649488.1373866354, bottom=5725865.471002353, right=1222320.6238982421, top=6085381.326011166)
NoData: -9999.0
Transform: | 291.81, 0.00, 649488.14|
| 0.00,-291.81, 6085381.33|
| 0.00, 0.00, 1.00|
